In [1]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import os
import json
import re
from transformers import AutoTokenizer

[nltk_data] Downloading package punkt to /u/adityav/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /u/adityav/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
def get_all_data():
    with open('/var/local/adityav/Projects/temp/mlwb/final_project/transcript_books.json', 'r') as f:
        all_data = json.load(f)
    return all_data

def sentence_based_chunking(text, max_sentences):
    if type(text) == list:
        text = '\n'.join(text)
    sentences = nltk.sent_tokenize(text)
    chunks = []
    current_chunk = []
    
    for sentence in sentences:
        if len(current_chunk) < max_sentences:
            current_chunk.append(sentence)
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
    
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks


def token_based_chunking(text, max_tokens):
    if type(text) == list:
        text = '\n'.join(text)
        
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    tokens = tokenizer.tokenize(text)
    chunks = []
    current_chunk = []
    
    for token in tokens:
        if len(current_chunk) < max_tokens:
            current_chunk.append(token)
        else:
            chunks.append(tokenizer.convert_tokens_to_string(current_chunk))
            current_chunk = [token]
    
    if current_chunk:
        chunks.append(tokenizer.convert_tokens_to_string(current_chunk))
    
    return chunks

In [6]:
def save_chunks(all_data, save_file):
    all_chunks = []
    ctr = 1
    for data_type, data in all_data.items():
        for d in data:
            all_chunks.append({'id': f"chunk_{ctr}", 'chunk': d})
            ctr += 1
    with open(save_file, 'w') as f:
        json.dump(all_chunks, f)

In [7]:
all_data = get_all_data()
print(all_data.keys())

dict_keys(['merged_transcript.txt', 'Transcript_Feb19_Mar5.txt', 'Bishop-Pattern-Recognition-and-Machine-Learning-2006.txt', 'understanding-machine-learning-theory-algorithms.txt', 'ESLII_print12_toc.txt', 'ProbabilisticMachineLearningAnIntroduction.txt', 'ProbabilisticMachineLearningAdvancedTopics.txt', 'ISLP_website.txt'])


In [65]:
outfile = "chunked_transcript_books_10sentences.json"
chunked_data = {}
total = 0
for data_type, data in all_data.items():
    print(f"Chunking file {data_type}")
    chunks = sentence_based_chunking(all_data[data_type], 10)
    print(f"Chunks: {len(chunks)}")
    chunked_data[data_type] = chunks
    total += len(chunks)
print(f"Total chunks: {total}")
# with open(outfile, 'w') as f:
#     json.dump(chunked_data, f)
save_chunks(chunked_data, outfile)

Chunking file merged_transcript.txt
Chunks: 690
Chunking file Transcript_Feb19_Mar5.txt
Chunks: 321
Chunking file Bishop-Pattern-Recognition-and-Machine-Learning-2006.txt
Chunks: 1179
Chunking file understanding-machine-learning-theory-algorithms.txt
Chunks: 887
Chunking file ESLII_print12_toc.txt
Chunks: 1274
Chunking file ProbabilisticMachineLearningAnIntroduction.txt
Chunks: 1484
Chunking file ProbabilisticMachineLearningAdvancedTopics.txt
Chunks: 2410
Chunking file ISLP_website.txt
Chunks: 1212
Total chunks: 9457


In [8]:
outfile = "chunked_transcript_books_1024berttokens.json"
chunked_data = {}
total = 0
for data_type, data in all_data.items():
    print(f"Chunking file {data_type}")
    chunks = token_based_chunking(all_data[data_type], 1024)
    print(f"Chunks: {len(chunks)}")
    chunked_data[data_type] = chunks
    total += len(chunks)
print(f"Total chunks: {total}")
# with open(outfile, 'w') as f:
#     json.dump(chunked_data, f)

save_chunks(chunked_data, outfile)

Chunking file merged_transcript.txt


Token indices sequence length is longer than the specified maximum sequence length for this model (83802 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (37859 > 512). Running this sequence through the model will result in indexing errors


Chunks: 82
Chunking file Transcript_Feb19_Mar5.txt
Chunks: 37
Chunking file Bishop-Pattern-Recognition-and-Machine-Learning-2006.txt


Token indices sequence length is longer than the specified maximum sequence length for this model (410264 > 512). Running this sequence through the model will result in indexing errors


Chunks: 401
Chunking file understanding-machine-learning-theory-algorithms.txt


...

Chunks: 461
Chunking file ProbabilisticMachineLearningAdvancedTopics.txt


Token indices sequence length is longer than the specified maximum sequence length for this model (825011 > 512). Running this sequence through the model will result in indexing errors


Chunks: 806
Chunking file ISLP_website.txt


Token indices sequence length is longer than the specified maximum sequence length for this model (356424 > 512). Running this sequence through the model will result in indexing errors


Chunks: 349
Total chunks: 2889


In [54]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
semantic_chunker = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")

In [60]:
semantic_chunks = semantic_chunker.create_documents('\n'.join(all_data['merged_transcript.txt']))
print(len(semantic_chunks))

341899


In [61]:
semantic_chunks

[Document(metadata={}, page_content=' '),
 Document(metadata={}, page_content='E'),
 Document(metadata={}, page_content='n'),
 Document(metadata={}, page_content='g'),
 Document(metadata={}, page_content='l'),
 Document(metadata={}, page_content='i'),
 Document(metadata={}, page_content='s'),
 Document(metadata={}, page_content='h'),
 Document(metadata={}, page_content='.'),
 Document(metadata={}, page_content='\n'),
 Document(metadata={}, page_content=' '),
 Document(metadata={}, page_content='O'),
 Document(metadata={}, page_content='k'),
 Document(metadata={}, page_content='a'),
 Document(metadata={}, page_content='y'),
 Document(metadata={}, page_content=','),
 Document(metadata={}, page_content=' '),
 Document(metadata={}, page_content='s'),
 Document(metadata={}, page_content='o'),
 Document(metadata={}, page_content=' '),
 Document(metadata={}, page_content='l'),
 Document(metadata={}, page_content='e'),
 Document(metadata={}, page_content='t'),
 Document(metadata={}, page_conte